In [1]:
import pandas as pd
import json
import os

# ============ 设置路径 ============
folder_path = os.getcwd()  # 使用当前工作目录

# 输入文件
csv_file = os.path.join(folder_path, 'all_sdg_data.csv')
un_file = os.path.join(folder_path, 'UNSDMethodology.csv')  # UN对照表

# 输出文件
output_json = os.path.join(folder_path, 'sdg_data_mapped.json')

print(" 开始转换 CSV → JSON")
print("="*60)

# ============ 1. 加载UN国家对照表 ============
print("\n📖 步骤1: 加载UN国家对照表...")
try:
    un_df = pd.read_csv(un_file, sep=';', encoding='utf-8')
    
    # 创建国家名 → ISO3 的映射字典
    country_to_iso = {}
    for _, row in un_df.iterrows():
        country_name = str(row['Country or Area']).strip()
        iso3 = str(row['ISO-alpha3 Code']).strip()
        
        if country_name and iso3 and iso3 != 'nan':
            country_to_iso[country_name] = iso3
    
    print(f"✓ 加载了 {len(country_to_iso)} 个国家的ISO映射")
    
except Exception as e:
    print(f" 加载UN对照表失败: {e}")
    print(" 请确保 UNSDMethodology.csv 在同一目录中")
    exit(1)

# ============ 2. 加载SDG数据 ============
print("\n 步骤2: 加载SDG数据...")
try:
    df = pd.read_csv(csv_file, encoding='utf-8-sig')
    print(f"✓ 加载了 {len(df)} 行数据")
except Exception as e:
    print(f" 加载CSV失败: {e}")
    exit(1)

# ============ 3. 转换为JSON格式 ============
print("\n 步骤3: 转换为JSON格式...")

result = {}
matched_count = 0
unmatched_countries = set()

for _, row in df.iterrows():
    country_name = row['Country']
    year = str(int(row['Year']))  # 确保年份是字符串
    sdg = int(row['SDG'])
    value = float(row['Value'])
    
    # 获取ISO代码
    iso3 = country_to_iso.get(country_name)
    
    if not iso3:
        unmatched_countries.add(country_name)
        continue
    
    matched_count += 1
    
    # 构建嵌套字典结构
    if iso3 not in result:
        result[iso3] = {}
    
    if year not in result[iso3]:
        result[iso3][year] = {}
    
    # 添加SDG数据
    sdg_key = f'sdg{sdg}'
    result[iso3][year][sdg_key] = value

print(f"✓ 成功转换: {matched_count}/{len(df)} 行")

# ============ 4. 显示未匹配的国家 ============
if unmatched_countries:
    print(f"\n  以下 {len(unmatched_countries)} 个国家/地区无法匹配ISO代码:")
    for i, country in enumerate(sorted(unmatched_countries), 1):
        print(f"   {i:2d}. {country}")

# ============ 5. 保存JSON文件 ============
print(f"\n 步骤4: 保存JSON文件...")
try:
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f"✓ JSON文件已保存: {output_json}")
except Exception as e:
    print(f" 保存失败: {e}")
    exit(1)

# ============ 6. 统计信息 ============
print("\n" + "="*60)
print(" 转换完成统计:")
print(f"   国家数量: {len(result)}")
print(f"   成功匹配: {matched_count} 行")
print(f"   无法匹配: {len(df) - matched_count} 行")

# 显示一些示例数据
print(f"\n JSON结构示例:")
sample_iso = list(result.keys())[0] if result else None
if sample_iso:
    sample_year = list(result[sample_iso].keys())[0]
    print(f'  "{sample_iso}": {{')
    print(f'    "{sample_year}": {{')
    for sdg_key, value in result[sample_iso][sample_year].items():
        print(f'      "{sdg_key}": {value}')
    print(f'    }}')
    print(f'  }}')

print("\n 转换完成！")

 开始转换 CSV → JSON

📖 步骤1: 加载UN国家对照表...
✓ 加载了 248 个国家的ISO映射

 步骤2: 加载SDG数据...
✓ 加载了 38867 行数据

 步骤3: 转换为JSON格式...
✓ 成功转换: 32872/38867 行

  以下 69 个国家/地区无法匹配ISO代码:
    1. Africa
    2. Americas
    3. Asia
    4. Australia and New Zealand
    5. Caribbean
    6. Caucasus and Central Asia
    7. Central America
    8. Central Asia
    9. Central and Southern Asia
   10. Channel Islands
   11. Côte d'Ivoire
   12. Developed regions (Europe, Cyprus, Israel, Northern America, Japan, Australia & New Zealand)
   13. Developing regions
   14. Eastern Africa
   15. Eastern Asia
   16. Eastern Asia (excluding Japan)
   17. Eastern Europe
   18. Eastern and South-Eastern Asia
   19. Europe
   20. Europe and Northern America
   21. FAO Major Fishing Area: Atlantic, Eastern Central
   22. FAO Major Fishing Area: Atlantic, Northeast
   23. FAO Major Fishing Area: Atlantic, Northwest
   24. FAO Major Fishing Area: Atlantic, Southeast
   25. FAO Major Fishing Area: Atlantic, Southwest
   26. FAO Major Fi